In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [3]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [4]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [5]:
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [6]:
ask("tell me about mechanical keyboard")

The mechanical keyboard is a high-quality product with a price of $129. It has a rating of 4.8 out of 5 stars, indicating excellent customer satisfaction. The keyboard features Cherry MX Brown switches, which are known for their tactile bump and smooth actuation. Additionally, it has per-key RGB lighting, allowing for customizable backlighting effects. The tenkeyless design makes it more compact and ideal for gamers or typists who prefer a more minimalist setup.


In [7]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [8]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [9]:
ask2("how do people like smart watch")

People like smart watches because they are convenient and provide a range of useful features such as tracking heart rate and sleep, and being water-resistant. The smart watch has a rating of 4.3 and is priced at $199.99. It also has a long battery life of 5 days. Additionally, according to reviews, people have given the smart watch an average rating of 3.9 out of 5, based on 340 reviews.


AGENT WITH MEMORY

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}}
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]},
            config
    )
    print(result["messages"][-1].content)

In [15]:
ask2("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [16]:
ask2("what are the reviews ont this product")

The wireless headphones have 1262 reviews with an average rating of 4.6.
